# 08 · Fishing pressure, stock resilience and recovery

**Atlas source reading copy · browser-network**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-Global-Fishing-Watch-and-Dark-Vessels/blob/145a73e500cb06070f399a040206802cc279174e/content/08_fish_stocks.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability.

**Provenance:** `content/08_fish_stocks.ipynb` at `145a73e500cb06070f399a040206802cc279174e`; original SHA-256 `77be7af69a48fc5c37020b9e411322317b67a88dec5f49b888a98ecdeb6a8a90`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# 08 · Fishing pressure, stock resilience and recovery

[Run this notebook in your browser ↗](https://jltobias.github.io/JupyterLite-Global-Fishing-Watch-and-Dark-Vessels/lab/index.html?path=08_fish_stocks.ipynb)

**Evidence: a published FAO summary and a separate synthetic biomass model.** Distinguish a stock assessment from a vessel activity map, and explore why growth and fishing mortality jointly shape recovery.

Source for observed summary: [FAO 2025 marine fish-stock assessment](https://www.fao.org/newsroom/detail/fao-releases-the-most-detailed-global-assessment-of-marine-fish-stocks-to-date). These figures describe that assessment, not an automatically updated estimate.

Run the setup cell first. Browser Python downloads numerical packages on first use. Core exercises use bundled files; live access is optional. Export your notebook to preserve work.

In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "pandas", "matplotlib", "scipy", "plotly"])
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import oceanlab as lab
from IPython.display import display
print("Runtime:", sys.version.split()[0], "| NumPy:", np.__version__, "| pandas:", pd.__version__)

## Two true percentages with different denominators
FAO's 2025 review reports 64.5% of assessed stocks at biologically sustainable levels and 77.2% of landings from biologically sustainable stocks when weighted by production. A stock-count percentage and a production-weighted percentage answer different questions. The assessment covers 2,570 stocks; it should not be mechanically appended to earlier assessment series without checking methodology.

In [ ]:
fao=json.loads((lab.DATA/"fao_2025_summary.json").read_text())
summary=pd.DataFrame({"indicator":["Stock-count share","Production-weighted landings share"],"sustainable_percent":[fao["sustainable_stock_count_percent"],fao["sustainable_production_weighted_landings_percent"]]})
display(summary)
ax=summary.plot.barh(x="indicator",y="sustainable_percent",color=["#087e83","#bd6e13"],legend=False)
ax.set(xlim=(0,100),xlabel="Percent (%)",ylabel="Denominator",title="OBSERVED FAO 2025 SUMMARY | denominators matter")
for y,v in enumerate(summary.sustainable_percent):ax.text(v+1,y,f"{v}%",va="center")
plt.tight_layout();plt.show()

## A deliberately simple population model
We use logistic growth with proportional fishing mortality:

`dB/dt = r B (1 − B/K) − F B`.

`B` is biomass, `r` intrinsic growth per year, `K` carrying capacity, and `F` fishing mortality per year. The model uses illustrative units. Apparent fishing hours are not `F`; converting effort to mortality would require catchability and species-specific evidence.

In [ ]:
lab.show(lab.stock_slider())

## More unobserved effort is an assumption to test
If total mortality were proportional to total effort, then `F_total = F_observed / (1 − hidden_share)`. That proportionality is a hypothetical scenario, not an inference from SAR unmatched fractions. The unmatched fractions in this book must not be substituted directly for `hidden_share`. Catchability, gear, species and behavior can differ.

In [ ]:
F_OBSERVED=.15
fig,ax=plt.subplots()
for share in [0,.25,.50,.70]:
    F_total=F_OBSERVED/(1-share)
    t,b=lab.stock_model(fishing_mortality=F_total)
    ax.plot(t,b,label=f"Hypothetical hidden effort {share:.0%}; F={F_total:.2f}")
ax.set(xlabel="Model year",ylabel="Biomass (illustrative units)",title="SYNTHETIC SCENARIOS | an assumption changes the trajectory")
ax.legend();plt.show()

## Recovery has conditions
In this model, a positive equilibrium exists when `F < r` and equals `K(1 − F/r)`. The model's maximum sustainable yield occurs at `B = K/2` and `F = r/2`. Those formulas are not management advice for a real stock. Real systems need uncertainty, stock structure, recruitment, climate, species interactions and socioeconomic evidence.

**Try it:** change `r` and initial biomass while holding `F` fixed. Then design a recovery scenario with a timed mortality reduction. Explain why improved vessel visibility alone does not guarantee recovery, nutrition, or fair access.